# Chapter 8.8: Privacy-Preserving Features for Recommendation

## Learning Objectives

By the end of this notebook, you will be able to:

1. Implement federated learning for recommendation (FedRec, FedMF)
2. Apply differential privacy (DP-SGD) to embedding training
3. Design on-device personalization with local model fine-tuning
4. Understand Apple's approach to privacy-first on-device recommendation
5. Explain secure aggregation and homomorphic encryption concepts
6. Navigate GDPR/CCPA implications for recommendation systems
7. Implement a federated collaborative filtering simulation

## Prerequisites

- Chapter 8.5: User modeling
- Chapter 8.7: Cross-domain basics
- Basic understanding of optimization (SGD)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hideak1/rec_system/blob/main/notebooks/part8/chapter_8.8_privacy.ipynb)
[![Download](https://img.shields.io/badge/Download-Notebook-blue)](https://github.com/hideak1/rec_system/raw/main/notebooks/part8/chapter_8.8_privacy.ipynb)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from typing import Dict, List, Tuple, Optional
from dataclasses import dataclass, field
from collections import defaultdict
import copy

torch.manual_seed(42)
np.random.seed(42)

plt.style.use('seaborn-v0_8')
print(f"PyTorch version: {torch.__version__}")

## 1. Federated Learning for Recommendation

In **federated learning (FL)**, user data never leaves the device. Instead:

1. Server sends the current global model to selected devices
2. Each device trains the model locally on its own data
3. Devices send model **updates** (gradients) back to the server
4. Server aggregates updates to improve the global model

For recommendation, this means:

$$\theta^{(t+1)} = \theta^{(t)} - \eta \cdot \frac{1}{|S|} \sum_{k \in S} \nabla \mathcal{L}_k(\theta^{(t)})$$

where $S$ is the set of participating devices and $\mathcal{L}_k$ is the local loss on device $k$.

### FedMF (Ammad-ud-din et al., 2019)
Federated Matrix Factorization: each user keeps their user embedding locally and only shares item embedding gradients.

> **\U0001f4a1 Concept:** The key privacy insight in FedMF is that user embeddings NEVER leave the device. Only aggregated item embedding updates are shared with the server, making it much harder to reconstruct individual user behavior.

In [ ]:
@dataclass
class ClientData:
    """Data stored on a single user's device."""
    user_id: int
    item_ids: List[int]
    ratings: List[float]


class FedMFClient:
    """Simulates a client device for Federated Matrix Factorization."""
    
    def __init__(self, user_id: int, embedding_dim: int, lr: float = 0.01):
        self.user_id = user_id
        self.embedding_dim = embedding_dim
        self.lr = lr
        
        # User embedding stays on device (private)
        self.user_embedding = torch.randn(embedding_dim) * 0.01
        self.user_bias = torch.zeros(1)
    
    def local_train(
        self,
        item_embeddings: torch.Tensor,
        item_biases: torch.Tensor,
        global_bias: float,
        local_data: ClientData,
        num_local_steps: int = 5,
    ) -> Tuple[Dict[int, torch.Tensor], Dict[int, torch.Tensor]]:
        """Train locally and return item embedding gradients (NOT user data)."""
        
        item_grad_accum = defaultdict(lambda: torch.zeros(self.embedding_dim))
        item_bias_grad_accum = defaultdict(lambda: torch.zeros(1))
        
        for step in range(num_local_steps):
            for item_id, rating in zip(local_data.item_ids, local_data.ratings):
                # Prediction
                item_emb = item_embeddings[item_id].clone()
                item_b = item_biases[item_id].clone()
                
                pred = (self.user_embedding * item_emb).sum() + self.user_bias + item_b + global_bias
                error = pred - rating
                
                # Compute gradients
                user_grad = error * item_emb + 0.01 * self.user_embedding
                item_grad = error * self.user_embedding + 0.01 * item_emb
                
                # Update user embedding locally (stays private)
                self.user_embedding -= self.lr * user_grad
                self.user_bias -= self.lr * error
                
                # Accumulate item gradients to send to server
                item_grad_accum[item_id] += item_grad
                item_bias_grad_accum[item_id] += error
        
        # Normalize by number of steps
        for k in item_grad_accum:
            item_grad_accum[k] /= num_local_steps
            item_bias_grad_accum[k] /= num_local_steps
        
        return dict(item_grad_accum), dict(item_bias_grad_accum)


class FedMFServer:
    """Federated MF server that aggregates client updates."""
    
    def __init__(self, num_items: int, embedding_dim: int, lr: float = 0.01):
        self.num_items = num_items
        self.embedding_dim = embedding_dim
        self.lr = lr
        
        # Item embeddings are on the server (public)
        self.item_embeddings = torch.randn(num_items, embedding_dim) * 0.01
        self.item_biases = torch.zeros(num_items)
        self.global_bias = 0.0
    
    def aggregate_updates(
        self,
        client_gradients: List[Dict[int, torch.Tensor]],
        client_bias_gradients: List[Dict[int, torch.Tensor]],
    ):
        """FedAvg-style aggregation of client gradients."""
        # Aggregate gradients for each item
        item_grad_sum = defaultdict(lambda: torch.zeros(self.embedding_dim))
        item_bias_sum = defaultdict(lambda: torch.zeros(1))
        item_counts = defaultdict(int)
        
        for client_grads, client_bias_grads in zip(client_gradients, client_bias_gradients):
            for item_id, grad in client_grads.items():
                item_grad_sum[item_id] += grad
                item_counts[item_id] += 1
            for item_id, grad in client_bias_grads.items():
                item_bias_sum[item_id] += grad
        
        # Apply averaged gradients
        for item_id in item_grad_sum:
            avg_grad = item_grad_sum[item_id] / item_counts[item_id]
            self.item_embeddings[item_id] -= self.lr * avg_grad
            avg_bias_grad = item_bias_sum[item_id] / item_counts[item_id]
            self.item_biases[item_id] -= self.lr * avg_bias_grad.squeeze()

# Generate synthetic data
num_users = 500
num_items = 200
embedding_dim = 16

# True user and item factors for data generation
true_user_factors = np.random.randn(num_users, 5) * 0.5
true_item_factors = np.random.randn(num_items, 5) * 0.5

# Generate client data
clients_data = []
for uid in range(num_users):
    # Each user rates some items
    num_ratings = np.random.randint(5, 30)
    rated_items = np.random.choice(num_items, size=num_ratings, replace=False).tolist()
    ratings = []
    for iid in rated_items:
        true_score = true_user_factors[uid] @ true_item_factors[iid] + 3.0
        rating = np.clip(true_score + np.random.randn() * 0.5, 1, 5)
        ratings.append(float(rating))
    clients_data.append(ClientData(uid, rated_items, ratings))

print(f"Generated data for {num_users} users, {num_items} items")
print(f"Avg ratings per user: {np.mean([len(c.ratings) for c in clients_data]):.1f}")

In [ ]:
# Run federated training
server = FedMFServer(num_items, embedding_dim, lr=0.01)
clients = [FedMFClient(uid, embedding_dim, lr=0.01) for uid in range(num_users)]

num_rounds = 30
clients_per_round = 50
round_losses = []

for round_idx in range(num_rounds):
    # Select random subset of clients
    selected = np.random.choice(num_users, size=clients_per_round, replace=False)
    
    # Local training
    all_item_grads = []
    all_bias_grads = []
    round_loss = 0
    
    for client_idx in selected:
        client = clients[client_idx]
        data = clients_data[client_idx]
        
        item_grads, bias_grads = client.local_train(
            server.item_embeddings,
            server.item_biases,
            server.global_bias,
            data,
            num_local_steps=3,
        )
        
        all_item_grads.append(item_grads)
        all_bias_grads.append(bias_grads)
        
        # Compute loss for monitoring
        for item_id, rating in zip(data.item_ids, data.ratings):
            pred = (client.user_embedding * server.item_embeddings[item_id]).sum() + \
                   client.user_bias + server.item_biases[item_id] + server.global_bias
            round_loss += (pred.item() - rating) ** 2
    
    # Server aggregation
    server.aggregate_updates(all_item_grads, all_bias_grads)
    
    avg_loss = round_loss / sum(len(clients_data[i].ratings) for i in selected)
    round_losses.append(avg_loss)
    
    if (round_idx + 1) % 10 == 0:
        print(f"Round {round_idx + 1}: avg MSE = {avg_loss:.4f}")

print(f"\nFinal MSE: {round_losses[-1]:.4f}")

## 2. Differential Privacy for Embedding Training (DP-SGD)

**DP-SGD** (Abadi et al., Google, 2016) modifies SGD to provide formal privacy guarantees:

1. **Clip** per-sample gradients to bound sensitivity: $\bar{g}_i = g_i / \max(1, \|g_i\| / C)$
2. **Add noise** proportional to the sensitivity: $\tilde{g} = \frac{1}{B}(\sum_i \bar{g}_i + \mathcal{N}(0, \sigma^2 C^2 I))$
3. **Privacy accounting**: Track cumulative privacy budget $(\epsilon, \delta)$

The privacy guarantee: For any two datasets differing in one user's data, the model's output distributions are $(\epsilon, \delta)$-indistinguishable.

$$\Pr[\mathcal{M}(D) \in S] \leq e^{\epsilon} \cdot \Pr[\mathcal{M}(D') \in S] + \delta$$

> **\u26a0\ufe0f Common Pitfall:** DP-SGD significantly slows convergence and reduces model quality. The noise level grows with the number of parameters, making it especially challenging for large embedding tables. Focus DP on the most sensitive features (user IDs) rather than all parameters.

In [ ]:
class DPSGDTrainer:
    """Simulates DP-SGD training for a recommendation model."""
    
    def __init__(
        self,
        model: nn.Module,
        lr: float = 0.01,
        clip_norm: float = 1.0,
        noise_multiplier: float = 1.0,
        delta: float = 1e-5,
    ):
        self.model = model
        self.lr = lr
        self.clip_norm = clip_norm
        self.noise_multiplier = noise_multiplier
        self.delta = delta
        self.optimizer = torch.optim.SGD(model.parameters(), lr=lr)
        
        # Privacy accounting
        self.steps = 0
        self.sample_rate = 0.01  # fraction of data per batch
    
    def _clip_and_noise_gradients(self, batch_size: int):
        """Clip gradients and add Gaussian noise."""
        # Compute gradient norm
        total_norm = 0.0
        for param in self.model.parameters():
            if param.grad is not None:
                total_norm += param.grad.data.norm(2).item() ** 2
        total_norm = total_norm ** 0.5
        
        # Clip
        clip_coeff = min(1.0, self.clip_norm / (total_norm + 1e-8))
        for param in self.model.parameters():
            if param.grad is not None:
                param.grad.data *= clip_coeff
        
        # Add noise
        noise_std = self.noise_multiplier * self.clip_norm / batch_size
        for param in self.model.parameters():
            if param.grad is not None:
                noise = torch.randn_like(param.grad) * noise_std
                param.grad.data += noise
        
        return total_norm, clip_coeff
    
    def train_step(self, loss: torch.Tensor, batch_size: int) -> Dict:
        self.optimizer.zero_grad()
        loss.backward()
        
        # Apply DP mechanism
        grad_norm, clip_coeff = self._clip_and_noise_gradients(batch_size)
        
        self.optimizer.step()
        self.steps += 1
        
        return {
            'loss': loss.item(),
            'grad_norm': grad_norm,
            'clip_coeff': clip_coeff,
        }
    
    def estimate_epsilon(self, num_examples: int) -> float:
        """Rough epsilon estimate using simple composition."""
        # This is a simplified estimate; real systems use RDP or GDP accountants
        q = self.sample_rate  # sampling rate
        sigma = self.noise_multiplier
        T = self.steps
        
        # Simple composition bound
        epsilon_per_step = q * np.sqrt(2 * np.log(1.25 / self.delta)) / sigma
        total_epsilon = epsilon_per_step * np.sqrt(T)  # Advanced composition
        return total_epsilon

# Simple recommendation model
class SimpleMF(nn.Module):
    def __init__(self, num_users, num_items, dim):
        super().__init__()
        self.user_emb = nn.Embedding(num_users, dim)
        self.item_emb = nn.Embedding(num_items, dim)
        nn.init.xavier_uniform_(self.user_emb.weight)
        nn.init.xavier_uniform_(self.item_emb.weight)
    
    def forward(self, users, items):
        return (self.user_emb(users) * self.item_emb(items)).sum(dim=-1)

# Compare training with different noise levels
noise_levels = [0.0, 0.5, 1.0, 2.0]
all_losses = {}

# Prepare data
all_users = []
all_items = []
all_ratings = []
for cd in clients_data:
    for iid, r in zip(cd.item_ids, cd.ratings):
        all_users.append(cd.user_id)
        all_items.append(iid)
        all_ratings.append(r)

train_u = torch.tensor(all_users)
train_i = torch.tensor(all_items)
train_r = torch.tensor(all_ratings, dtype=torch.float32)

for noise in noise_levels:
    model = SimpleMF(num_users, num_items, embedding_dim)
    trainer = DPSGDTrainer(model, lr=0.05, clip_norm=1.0, noise_multiplier=max(noise, 1e-10))
    
    losses = []
    batch_size = 128
    
    for epoch in range(20):
        perm = torch.randperm(len(train_u))
        epoch_loss = 0
        n = 0
        for i in range(0, len(train_u), batch_size):
            idx = perm[i:i+batch_size]
            preds = model(train_u[idx], train_i[idx])
            loss = F.mse_loss(preds, train_r[idx])
            
            if noise > 0:
                stats = trainer.train_step(loss, len(idx))
            else:
                trainer.optimizer.zero_grad()
                loss.backward()
                trainer.optimizer.step()
            
            epoch_loss += loss.item()
            n += 1
        
        losses.append(epoch_loss / n)
    
    all_losses[noise] = losses
    eps = trainer.estimate_epsilon(len(train_u)) if noise > 0 else float('inf')
    print(f"Noise={noise:.1f}: Final MSE={losses[-1]:.4f}, Estimated epsilon={eps:.2f}")

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
for noise, losses in all_losses.items():
    label = f'sigma={noise}' if noise > 0 else 'No DP'
    ax.plot(losses, label=label, linewidth=2)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('MSE Loss', fontsize=12)
ax.set_title('DP-SGD: Privacy-Utility Trade-off', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig('dp_sgd_training.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. On-Device Personalization

**On-device personalization** (Apple ML, 2019) keeps personal data entirely on the user's device:

1. A **base model** is trained centrally on aggregated, anonymized data
2. Each device receives the base model
3. The device **fine-tunes** the model locally using on-device data
4. No personal data or model updates leave the device

This is Apple's approach for:
- Siri suggestions
- News recommendations
- App suggestions

> **\U0001f511 Pro Tip:** The base model should be small enough to run efficiently on mobile devices (typically <50MB). Use knowledge distillation to create a compact base model from a larger server model.

In [ ]:
class OnDevicePersonalizer:
    """Simulates on-device model personalization."""
    
    def __init__(
        self,
        base_model: nn.Module,
        personalize_layers: List[str],  # Names of layers to fine-tune
        lr: float = 0.001,
        max_local_samples: int = 100,  # On-device memory limit
    ):
        self.base_model = copy.deepcopy(base_model)
        self.personalize_layers = personalize_layers
        self.lr = lr
        self.max_local_samples = max_local_samples
        
        # Freeze non-personalized layers
        for name, param in self.base_model.named_parameters():
            should_train = any(layer_name in name for layer_name in personalize_layers)
            param.requires_grad = should_train
        
        trainable = sum(p.numel() for p in self.base_model.parameters() if p.requires_grad)
        total = sum(p.numel() for p in self.base_model.parameters())
        self.trainable_ratio = trainable / total
        
        self.optimizer = torch.optim.Adam(
            [p for p in self.base_model.parameters() if p.requires_grad],
            lr=lr
        )
        
        # Local data buffer (FIFO)
        self.local_data: List[Tuple[torch.Tensor, torch.Tensor, torch.Tensor]] = []
    
    def add_local_sample(self, user_id: int, item_id: int, label: float):
        """Add an interaction to local on-device data."""
        self.local_data.append((
            torch.tensor([user_id]),
            torch.tensor([item_id]),
            torch.tensor([label]),
        ))
        # Enforce memory limit
        if len(self.local_data) > self.max_local_samples:
            self.local_data.pop(0)
    
    def personalize(self, num_steps: int = 10) -> float:
        """Fine-tune on local data."""
        if len(self.local_data) < 5:
            return float('inf')
        
        self.base_model.train()
        total_loss = 0
        
        for step in range(num_steps):
            # Sample mini-batch from local data
            idx = np.random.choice(len(self.local_data), size=min(16, len(self.local_data)), replace=False)
            users = torch.cat([self.local_data[i][0] for i in idx])
            items = torch.cat([self.local_data[i][1] for i in idx])
            labels = torch.cat([self.local_data[i][2] for i in idx])
            
            preds = self.base_model(users, items)
            loss = F.mse_loss(preds, labels)
            
            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()
            total_loss += loss.item()
        
        return total_loss / num_steps
    
    def predict(self, user_id: int, item_ids: torch.Tensor) -> torch.Tensor:
        self.base_model.eval()
        with torch.no_grad():
            users = torch.tensor([user_id]).expand(len(item_ids))
            return self.base_model(users, item_ids)

# Train a centralized base model
base_model = SimpleMF(num_users, num_items, embedding_dim)
base_optimizer = torch.optim.Adam(base_model.parameters(), lr=0.01)

for epoch in range(20):
    perm = torch.randperm(len(train_u))
    for i in range(0, len(train_u), 256):
        idx = perm[i:i+256]
        pred = base_model(train_u[idx], train_i[idx])
        loss = F.mse_loss(pred, train_r[idx])
        base_optimizer.zero_grad()
        loss.backward()
        base_optimizer.step()

# Test on-device personalization for a specific user
test_user_id = 0
personalizer = OnDevicePersonalizer(
    base_model, 
    personalize_layers=['user_emb'],  # Only fine-tune user embedding
    lr=0.01
)

print(f"Trainable parameters: {personalizer.trainable_ratio*100:.1f}% of model")

# Add user's interactions to device
for iid, r in zip(clients_data[test_user_id].item_ids, clients_data[test_user_id].ratings):
    personalizer.add_local_sample(test_user_id, iid, r)

# Compare before and after personalization
test_items = torch.tensor(clients_data[test_user_id].item_ids[:10])
test_ratings = torch.tensor(clients_data[test_user_id].ratings[:10])

before_preds = personalizer.predict(test_user_id, test_items)
before_mse = F.mse_loss(before_preds, test_ratings).item()

final_loss = personalizer.personalize(num_steps=50)

after_preds = personalizer.predict(test_user_id, test_items)
after_mse = F.mse_loss(after_preds, test_ratings).item()

print(f"\nUser {test_user_id} personalization:")
print(f"  Before: MSE = {before_mse:.4f}")
print(f"  After:  MSE = {after_mse:.4f}")
print(f"  Improvement: {(1 - after_mse/before_mse)*100:.1f}%")

## 4. Secure Aggregation Overview

**Secure aggregation** ensures the server can compute the average of client updates without seeing any individual update.

Protocol sketch (Bonawitz et al., Google, 2017):
1. Each client $k$ generates a random mask $r_k$
2. Client sends $g_k + r_k$ instead of $g_k$
3. Masks are designed so that $\sum_k r_k = 0$
4. Server computes $\sum_k (g_k + r_k) = \sum_k g_k$

Individual $g_k$ is hidden, but the sum is exact.

Let's simulate this concept.

In [ ]:
class SecureAggregationSimulator:
    """Simulates secure aggregation protocol."""
    
    def __init__(self, num_clients: int, gradient_dim: int):
        self.num_clients = num_clients
        self.gradient_dim = gradient_dim
    
    def generate_masks(self) -> List[torch.Tensor]:
        """Generate random masks that sum to zero."""
        masks = []
        running_sum = torch.zeros(self.gradient_dim)
        
        for i in range(self.num_clients - 1):
            mask = torch.randn(self.gradient_dim) * 100  # Large noise
            masks.append(mask)
            running_sum += mask
        
        # Last mask ensures sum is zero
        masks.append(-running_sum)
        return masks
    
    def secure_aggregate(
        self, client_gradients: List[torch.Tensor]
    ) -> Tuple[torch.Tensor, Dict]:
        """Perform secure aggregation."""
        masks = self.generate_masks()
        
        # What the server sees (masked gradients)
        masked_gradients = [
            g + m for g, m in zip(client_gradients, masks)
        ]
        
        # Server sums masked gradients
        server_sum = torch.stack(masked_gradients).sum(dim=0)
        
        # True sum (for verification)
        true_sum = torch.stack(client_gradients).sum(dim=0)
        
        # Stats
        info = {
            'reconstruction_error': (server_sum - true_sum).abs().max().item(),
            'avg_mask_magnitude': np.mean([m.abs().mean().item() for m in masks]),
            'avg_gradient_magnitude': np.mean([g.abs().mean().item() for g in client_gradients]),
            'individual_hidden': all(
                (mg - g).abs().mean().item() > g.abs().mean().item()
                for mg, g in zip(masked_gradients, client_gradients)
            ),
        }
        
        return server_sum / self.num_clients, info

# Demo secure aggregation
num_clients = 10
grad_dim = 64

sa = SecureAggregationSimulator(num_clients, grad_dim)

# Simulate client gradients
client_grads = [torch.randn(grad_dim) * 0.1 for _ in range(num_clients)]

avg_grad, info = sa.secure_aggregate(client_grads)

print("Secure Aggregation Results:")
print(f"  Reconstruction error: {info['reconstruction_error']:.2e}")
print(f"  Avg mask magnitude: {info['avg_mask_magnitude']:.2f}")
print(f"  Avg gradient magnitude: {info['avg_gradient_magnitude']:.4f}")
print(f"  Individual gradients hidden: {info['individual_hidden']}")
print(f"  Signal-to-noise ratio: {info['avg_gradient_magnitude']/info['avg_mask_magnitude']:.6f}")

## 5. GDPR/CCPA Implications for Recommendation Systems

Key regulatory requirements:

| Requirement | GDPR | CCPA | Impact on RecSys |
|-------------|------|------|------------------|
| Right to deletion | Art. 17 | Sec. 1798.105 | Must delete user embeddings on request |
| Data minimization | Art. 5(1)(c) | - | Don't collect more than needed |
| Purpose limitation | Art. 5(1)(b) | - | Features for one purpose can't be reused |
| Right to explanation | Art. 22 | - | Must explain automated decisions |
| Consent | Art. 6 | Sec. 1798.120 | Opt-in (GDPR) or opt-out (CCPA) |

> **\u26a0\ufe0f Common Pitfall:** The right to deletion is particularly tricky for recommendation systems. Deleting a user's embedding from the model doesn't remove their influence on other users' embeddings trained on shared interactions. True "machine unlearning" is an active research area.

In [ ]:
class GDPRCompliantRecSystem:
    """Recommendation system with GDPR compliance features."""
    
    def __init__(self, model: SimpleMF, num_users: int, num_items: int):
        self.model = model
        self.num_users = num_users
        self.num_items = num_items
        
        # Consent tracking
        self.consent_given = set()  # user_ids that gave consent
        self.data_purposes = defaultdict(set)  # user_id -> set of purposes
        
        # Deletion log
        self.deletion_log = []
    
    def give_consent(self, user_id: int, purpose: str):
        self.consent_given.add(user_id)
        self.data_purposes[user_id].add(purpose)
    
    def check_consent(self, user_id: int, purpose: str) -> bool:
        return user_id in self.consent_given and purpose in self.data_purposes[user_id]
    
    def delete_user_data(self, user_id: int) -> Dict:
        """Exercise right to deletion (Art. 17 GDPR)."""
        if user_id >= self.num_users:
            return {'status': 'error', 'message': 'User not found'}
        
        # Reset user embedding to random initialization
        with torch.no_grad():
            old_emb = self.model.user_emb.weight[user_id].clone()
            self.model.user_emb.weight[user_id] = torch.randn_like(old_emb) * 0.01
        
        # Remove consent
        self.consent_given.discard(user_id)
        if user_id in self.data_purposes:
            del self.data_purposes[user_id]
        
        # Log deletion
        self.deletion_log.append({
            'user_id': user_id,
            'embedding_deleted': True,
            'consent_revoked': True,
        })
        
        return {
            'status': 'success',
            'embedding_similarity_before_after': F.cosine_similarity(
                old_emb.unsqueeze(0),
                self.model.user_emb.weight[user_id].unsqueeze(0)
            ).item()
        }
    
    def predict_with_consent_check(
        self, user_id: int, item_ids: torch.Tensor, purpose: str
    ) -> Optional[torch.Tensor]:
        """Only serve predictions if consent is given for this purpose."""
        if not self.check_consent(user_id, purpose):
            return None  # No consent -> no personalized recommendations
        
        with torch.no_grad():
            users = torch.tensor([user_id]).expand(len(item_ids))
            return self.model(users, item_ids)

# Demo
gdpr_system = GDPRCompliantRecSystem(base_model, num_users, num_items)

# User gives consent
gdpr_system.give_consent(0, 'personalized_recommendations')

# Predict with consent
items = torch.arange(0, 10)
preds = gdpr_system.predict_with_consent_check(0, items, 'personalized_recommendations')
print(f"With consent: predictions = {preds is not None}")

# Predict without consent for different purpose
preds_ad = gdpr_system.predict_with_consent_check(0, items, 'targeted_advertising')
print(f"Without consent (ads): predictions = {preds_ad is not None}")

# Exercise right to deletion
result = gdpr_system.delete_user_data(0)
print(f"\nDeletion result: {result}")
print(f"Deletion log: {gdpr_system.deletion_log}")

In [ ]:
# Visualize federated learning convergence and privacy comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Federated training convergence
ax1.plot(round_losses, linewidth=2, color='steelblue')
ax1.set_xlabel('Communication Round', fontsize=12)
ax1.set_ylabel('MSE Loss', fontsize=12)
ax1.set_title('Federated MF Training Convergence', fontsize=13, fontweight='bold')
ax1.annotate(f'Final MSE: {round_losses[-1]:.3f}', 
            xy=(len(round_losses)-1, round_losses[-1]),
            xytext=(-60, 20), textcoords='offset points',
            fontsize=10, arrowprops=dict(arrowstyle='->', color='black'))

# Plot 2: Privacy methods comparison
methods = ['Centralized\n(No Privacy)', 'Federated\nLearning', 'DP-SGD\n(sigma=1)', 'On-Device\nOnly']
privacy_score = [1, 3, 4, 5]  # Higher = more private
utility_score = [5, 4, 3, 3.5]  # Higher = better utility

x = np.arange(len(methods))
width = 0.35

bars1 = ax2.bar(x - width/2, privacy_score, width, label='Privacy', color='steelblue', edgecolor='black')
bars2 = ax2.bar(x + width/2, utility_score, width, label='Utility', color='coral', edgecolor='black')

ax2.set_ylabel('Score (1-5)', fontsize=12)
ax2.set_title('Privacy vs Utility Across Methods', fontsize=13, fontweight='bold')
ax2.set_xticks(x)
ax2.set_xticklabels(methods, fontsize=9)
ax2.legend(fontsize=11)
ax2.set_ylim(0, 6)

plt.tight_layout()
plt.savefig('privacy_methods.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Exercises

### \U0001f3cb\ufe0f Exercise 1: Implement FedRec with Privacy

Combine federated learning with differential privacy for the strongest guarantee.

In [ ]:
# Exercise 1: Federated Learning + Differential Privacy
# TODO: Implement FedMF with DP noise added to client updates

class DPFedMFClient(FedMFClient):
    """
    Federated MF client that adds DP noise to gradient updates
    before sending to the server.
    
    Steps:
    1. Compute local gradients (same as FedMFClient)
    2. Clip each gradient to max_norm
    3. Add Gaussian noise proportional to max_norm * noise_multiplier
    4. Send noisy gradients to server
    """
    
    def __init__(self, user_id: int, embedding_dim: int, lr: float = 0.01,
                 clip_norm: float = 1.0, noise_multiplier: float = 0.5):
        super().__init__(user_id, embedding_dim, lr)
        # TODO: Store DP parameters
        pass
    
    def local_train(
        self, item_embeddings, item_biases, global_bias, local_data, num_local_steps=5
    ):
        # TODO: Call parent's local_train, then add DP noise to returned gradients
        pass

# Compare: FedMF vs FedMF+DP at different noise levels

### \U0001f3cb\ufe0f Exercise 2: Machine Unlearning

Implement approximate machine unlearning for when a user requests deletion.

In [ ]:
# Exercise 2: Machine Unlearning
# TODO: Implement approximate machine unlearning

class UnlearnableRecSystem:
    """
    When a user requests deletion:
    1. Remove their embedding
    2. Fine-tune the model to "forget" their influence on item embeddings
    3. Verify that the model behaves as if the user never existed
    
    Approach: Take a few gradient steps to move item embeddings away from
    the deleted user's influence (approximate Newton step).
    """
    
    def __init__(self, model: SimpleMF):
        # TODO: Initialize
        pass
    
    def unlearn_user(
        self, user_id: int, user_data: ClientData, num_steps: int = 10
    ) -> Dict:
        # TODO: Implement approximate unlearning
        # 1. Delete user embedding
        # 2. For each item the user interacted with,
        #    take gradient steps to reverse the user's influence
        pass
    
    def verify_unlearning(self, user_id: int) -> Dict:
        # TODO: Check that the model's predictions for this user
        # are no better than random
        pass

### \U0001f3cb\ufe0f Exercise 3: Privacy Budget Manager

Build a system that tracks privacy budget expenditure across multiple model updates.

In [ ]:
# Exercise 3: Privacy Budget Manager
# TODO: Track cumulative privacy expenditure

class PrivacyBudgetManager:
    """
    Tracks the privacy budget (epsilon, delta) across multiple
    model training rounds and queries.
    
    Uses basic composition theorem:
    - Sequential composition: epsilons add up
    - Advanced composition: epsilons grow as sqrt(k)
    
    Alerts when budget is about to be exhausted.
    """
    
    def __init__(self, total_epsilon_budget: float, total_delta_budget: float):
        # TODO: Initialize budget tracking
        pass
    
    def record_query(self, epsilon: float, delta: float, description: str):
        # TODO: Record a privacy-consuming operation
        pass
    
    def remaining_budget(self) -> Tuple[float, float]:
        # TODO: Return remaining (epsilon, delta)
        pass
    
    def can_afford(self, epsilon: float, delta: float) -> bool:
        # TODO: Check if we can afford another query
        pass
    
    def get_report(self) -> Dict:
        # TODO: Return spending report
        pass

# Test: simulate a week of model updates and check budget

## Summary

In this notebook, we explored privacy-preserving techniques for recommendation:

| Technique | Privacy Level | Utility Impact | Deployment Complexity |
|-----------|-------------|----------------|----------------------|
| Federated Learning | High | Low-Medium | High |
| DP-SGD | Configurable | Medium-High | Medium |
| On-Device Personalization | Very High | Low | Medium |
| Secure Aggregation | High | None | Very High |
| Data Deletion | Compliance | Low | Medium |

### Key References

- McMahan et al. "Communication-Efficient Learning of Deep Networks (FedAvg)" (Google, AISTATS 2017)
- Abadi et al. "Deep Learning with Differential Privacy" (Google, CCS 2016)
- Ammad-ud-din et al. "Federated Collaborative Filtering for On-Device Next App Prediction" (2019)
- Bonawitz et al. "Practical Secure Aggregation" (Google, CCS 2017)
- Apple Machine Learning Research. "On-Device Personalization" (2019)
- Bourtoule et al. "Machine Unlearning" (IEEE S&P, 2021)

### Congratulations!

You've completed Part 8: Industrial Embedding & Feature Systems. You now have a comprehensive understanding of how modern recommendation systems manage embeddings, features, and user/item representations at scale, while navigating privacy and regulatory requirements.